# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

md = dataset.metadata  # md is the metadata object

print(f"{md.name}: {md.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. This allows you to understand what tables and columns are available in the dataset.

In [ ]:
# List all Record Sets (tables) and summarize their fields
record_sets = md.record_set  # list of RecordSet objects
print("Available Record Sets and their Field @ids:")
for rs in record_sets:
    print(f"- Record Set: {rs.name}")
    print(f"  @id: {rs['@id']}")
    print(f"  Fields:")
    for field in rs.field:
        print(f"    - {field.name} (@id: {field['@id']}) Type: {field.data_type if hasattr(field,'data_type') else 'unknown'}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the Record Set and Field `@id`s from the overview.

> _For this notebook, we'll extract from the main record set which contains protein information._

In [ ]:
# Typically, biological protein data is in the first or main record set. Let's choose the first Record Set.
# If there are multiple, this can be adapted to process each.

# Get list of record sets and their @ids
record_sets = md.record_set
record_set_ids = [rs['@id'] for rs in record_sets]
print(f"Record set @ids found: {record_set_ids}")

# We'll work with the first available record set for demonstration
protein_record_set_id = record_set_ids[0]
print(f"Using record set: {protein_record_set_id}")

dataframes = {}

records = list(dataset.records(record_set=protein_record_set_id))
df = pd.DataFrame(records)
dataframes[protein_record_set_id] = df

print("Data columns:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming distributions, or grouping data.

In [ ]:
# Example: We search for typical quantitative/abundance fields in the DataFrame

# Let's automatically pick a numeric field from the dataframe
numeric_fields = df.select_dtypes(include='number').columns.tolist()
print(f"Numeric fields available: {numeric_fields}")

# Use the first numeric field for this EDA
if numeric_fields:
    numeric_field = numeric_fields[0]
    print(f"Using numeric field for analysis: {numeric_field}")
else:
    numeric_field = None

if numeric_field:
    threshold = df[numeric_field].mean()  # Use the mean as an example threshold
    filtered_df = df[df[numeric_field] > threshold]

    print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
    display(filtered_df.head())

    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, norm_col]].head())

    # Attempt grouping by an available string/categorical field
    group_fields = df.select_dtypes(include='object').columns.tolist()
    if group_fields:
        group_field = group_fields[0]
        print(f"Grouping by: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"Grouped data by {group_field} (mean of {numeric_field}):")
        display(grouped_df.head())
    else:
        print("No string field found for grouping.")
else:
    print("No numeric field was detected for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We use matplotlib for plotting Example: histogram of abundances.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If grouping possible, plot means
    if 'grouped_df' in locals() and not grouped_df.empty:
        grouped_df = grouped_df.sort_values(by=numeric_field, ascending=False)
        plt.figure(figsize=(10, 5))
        grouped_df.head(15).plot(kind='bar', legend=False)
        plt.title(f"Mean {numeric_field} by {group_field} (top 15)")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(group_field)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to explore a mass spectrometry dataset of extracellular vesicle proteins using the `mlcroissant` library. We loaded the Croissant metadata, listed record sets and fields (by `@id`), extracted records, and performed basic exploratory data analysis and visualization using pandas and matplotlib/seaborn.

This workflow can be extended to more advanced protein filtering, normalization, or cross-sample comparison based on the available schema and data columns. For further analyses, consult the dataset documentation or the Croissant schema.